# Public Key Cryptography

Learn asymmetric encryption using RSA public/private key pairs.

In [1]:
from pathlib import Path

from Crypto.Cipher import PKCS1_OAEP
from Crypto.PublicKey import RSA      #public/private key generation

# Search for key files starting from current directory and moving up
def find_project_root():
    current = Path.cwd()
    # Try current directory first
    if (current / "my_keypair").exists():
        return current
    # Try parent directory
    if (current.parent / "my_keypair").exists():
        return current.parent
    # Try two levels up (for nested structures)
    if (current.parent.parent / "my_keypair").exists():
        return current.parent.parent
    return current  # Default to current if not found

PROJECT_FOLDER = find_project_root()
print(f"Looking for keypairs in folder: {PROJECT_FOLDER}")

PRIVATE_KEY_FILE = PROJECT_FOLDER / "my_keypair"
PUBLIC_KEY_FILE = PROJECT_FOLDER / "my_keypair.pub"

assert PRIVATE_KEY_FILE.exists(), f"Private key not found at {PRIVATE_KEY_FILE}. CWD: {Path.cwd()}"
assert PUBLIC_KEY_FILE.exists(), f"Public key not found at {PUBLIC_KEY_FILE}"

print(f"✓ Found private key: {PRIVATE_KEY_FILE}")
print(f"✓ Found public key: {PUBLIC_KEY_FILE}")


Looking for keypairs in folder: c:\Users\Admin\ceu\ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services
✓ Found private key: c:\Users\Admin\ceu\ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services\my_keypair
✓ Found public key: c:\Users\Admin\ceu\ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services\my_keypair.pub


Load the private key from file
The private key must be kept secret and secure
Extract the public key from our private key
The public key can be freely shared with anyone


In [2]:
with open(PRIVATE_KEY_FILE, "r", encoding="utf8") as key_file:
    private_key = RSA.import_key(key_file.read())

public_key = private_key.publickey()
print(f"Public key:\n{public_key.export_key().decode('utf-8')}")


Public key:
-----BEGIN PUBLIC KEY-----
MIIBojANBgkqhkiG9w0BAQEFAAOCAY8AMIIBigKCAYEA1gy94kyNbOxQatuBRrcP
8YQ4ja6F2Madi8DKjcoluARpu10p7qi/eZqWygEhZFy8mOi5d8+aTpETZdeOCY8k
RqbzC/Z5Xd9zR4pBZGQgqJ5RXm0zVFQ+Mb6H7sFuJsPPPdNWFiwviGQjRSQy+tqy
2g6gMj4mIGZMEkTUUR6jn0l0XO2xZtqg8PsbW8ARVQty9hynwB1nf5LNCI1HyzzR
VSL73pd8UPgp7mUl13vodg2a9GxbBy956r54kP69ZC+laRkllAz60K1k7SLSBs5c
plKsucl7jItYMVnFujUL3iRODmgAbXqHlLYlQMqTD9snk76oyocNgdigqRHWcgzd
i2gXIXUIwS8DVB3HmvROrtgei+7Z6IZHhb5fXN0Fox9O4Ynj9rkC+HKNIkhsa+Ia
eskUIqC+ZE6tgLDgv/4BqfpTx4QyzAqZYnK/QYWeKs2IrOEO4CVTwmqNnDoGMnHV
I/3G6Bnk/vXvYa9WCmRlggDxmN5P+QMupfoGZimWs1mVAgMBAAE=
-----END PUBLIC KEY-----


Message to be encrypted - must be converted to bytes
Create a cipher object using the public key for encryption
Encrypt our message - only someone with the private key can decrypt it
Save the encrypted message to a file


In [3]:
short_secret_message = "My Secret Message".encode("utf-8")

public_key_cipher = PKCS1_OAEP.new(public_key)

encrypted_message = public_key_cipher.encrypt(short_secret_message)
print("Encrypted message:")
print(encrypted_message.hex())

ENCRYPTED_MESSAGE_FILE = PROJECT_FOLDER / "encrypted_message.bin"
with open(ENCRYPTED_MESSAGE_FILE, "wb") as f:
    f.write(encrypted_message)


Encrypted message:
a9586adf596efa76bf4aa11a00421f527bdd0c6f838a555b0019933062d2e0ef08e581154542eb86ca4c4d19a43b3337cefe78d39bf9abeef57a7904fdda201ab7f136fd3f569386d306582b15cf4aa79502d6901dfe17626c806caa6b1a03826f8f3f60a33a408c124eab8ed63a587c2667b17918acf2cf8d51f87abf78ef4a235c89243d259529da5d8af5900b5d80e6f317f9fff283a8651482232f7a3e45d0cca7e6a35a2d42bbfa331a8e185475700c0aa9a6ab9ffeccf6a1f755c59b6680d62b46965c1277904b02c85ae41e98ce2a7d1b6dfd401b3ca1b01d4d2c29b2d26b90ccc97fc0cfd68bd399b0e4bd41580901ab8fd583436e3e7e8b09e6901f776dd67d7af67675bf1c443fadfe11db425cbe36ccf0356a4650b4ca8d8c87fbc05a54dfc0af2137e915e7c5a3f4a78b2d799c87033ea87ed2ce4773e92ea8902a2d50b3a3c036a327bb2faf36c3e2ee4fe31d3b192cfdba4d05e63ea6fae8d106ae194499bd57ce28059b59eae9fa18f7d7ec12ff784f470cfc24773c14f2e2


Read the encrypted message from file
Create a cipher object using the private key for decryption
Decrypt the message using the private key
This demonstrates asymmetric encryption where:
1. Messages are encrypted with a public key and can only be decrypted with the matching private key
2. The public key can be freely shared
3. The private key must be kept secret
4. This is more secure than symmetric encryption as the decryption key (private key) is never shared


In [4]:
with open(ENCRYPTED_MESSAGE_FILE, "rb") as f:
    encrypted_message_from_file = f.read()

private_key_cipher = PKCS1_OAEP.new(private_key)

decrypted_message = private_key_cipher.decrypt(encrypted_message_from_file)
print(f"Decrypted message: {decrypted_message.decode('utf-8')}")



Decrypted message: My Secret Message
